# ATML PA1 - Beyond IID

Runs the repository's scripts on a Colab GPU. **No experiment logic lives in this notebook** - every cell clones, configures, or invokes repository code.

Set **Runtime -> Change runtime type -> T4 GPU** before running.

Full details are in `COLAB.md` in the repository.

## 1. Verify GPU and mount Drive

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0), "|", torch.__version__)

from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/ATML_PA1'
!mkdir -p {DRIVE}/checkpoints {DRIVE}/results {DRIVE}/figures {DRIVE}/logs {DRIVE}/data
!ls {DRIVE}

## 2. Clone and install

torch/torchvision are preinstalled with a matching CUDA build. Installing `requirements.txt` wholesale would replace them with slower wheels, so only the extras are installed here.

In [ ]:
REPO = "https://github.com/TalhaShoyo10/PA1_28100131.git"

!git clone {REPO} /content/pa1
%cd /content/pa1
!git log --oneline -3

!pip install -q open_clip_torch umap-learn pyyaml

## 3. Run the test suite

Needs no datasets. Expect **381 passed, 1 skipped** (the skip requires AdaIN weights). If anything fails, stop and fix it before spending GPU time.

In [ ]:
!python -m pytest tests/ -q

## 4. Data

STL-10 and CIFAR download automatically. **PACS must be uploaded to Drive manually** - it has no torchvision loader.

Expected layout: `ATML_PA1/data/PACS/<domain>/<class>/*.jpg` for 4 domains and 7 classes.

In [ ]:
!ln -sfn {DRIVE}/data /content/pa1/data
!ls {DRIVE}/data/PACS 2>/dev/null || echo "PACS NOT FOUND - upload it to Drive first"

# Shared split manifest for Tasks 2 and 3. Generate ONCE: regenerating between
# tasks would invalidate checkpoints trained on the other split.
!python shared/make_splits.py --config task2/configs/base.yaml

## 5. Smoke tests

Always run these first. A path or protocol error surfaces in seconds instead of an hour into a real run.

**Never report a `--smoke` number** - they use 1 epoch and a handful of batches.

In [ ]:
!python task2/train.py --config task2/configs/source_only.yaml --smoke
!python task4/train.py --config task4/configs/vanilla.yaml --smoke

## 6. Task 2 - Unsupervised Domain Adaptation

About 4 GPU-hours including the sweep. **Start this first**: `source_only` is also Task 3's ERM baseline.

In [ ]:
D2 = f"output.checkpoint_dir={DRIVE}/checkpoints/task2 output.results_dir={DRIVE}/results/task2"

for m in ["source_only", "dan", "dann", "cdan"]:
    print(f"\n===== {m} =====")
    !python task2/train.py --config task2/configs/{m}.yaml --set {D2}

In [ ]:
# Controlled study: lambda_MMD in {0.1, 1, 10}. Analysis only - the main
# comparison stays at lambda=1 and may not be replaced by a post-hoc winner.
for lam in ["0.1", "1.0", "10.0"]:
    !python task2/train.py --config task2/configs/dan_lambda_study.yaml \
        --set method.lambda_mmd={lam} run_name=dan_lambda{lam} {D2}

In [ ]:
# Target labels are used ONLY here, after every checkpoint is frozen.
!python task2/evaluate_final.py --config task2/configs/base.yaml --set {D2}

## 7. Task 3 - Domain Generalization

About 4 GPU-hours. SAM is roughly 2x ERM per epoch, since it does two forward/backward passes.

ERM is **not retrained**: `task3/train.py` refuses `erm.yaml` and reuses Task 2's checkpoint.

In [ ]:
D3 = f"output.checkpoint_dir={DRIVE}/checkpoints/task3 output.results_dir={DRIVE}/results/task3"

!python task3/train.py --config task3/configs/dan_dg.yaml --set {D3}
!python task3/train.py --config task3/configs/sam.yaml    --set {D3}

for lam in ["0.1", "1.0", "10.0"]:
    !python task3/train.py --config task3/configs/dan_dg_lambda_study.yaml \
        --set method.lambda_dg={lam} run_name=dan_dg_lambda{lam} {D3}

In [ ]:
# The ONLY script permitted to load Sketch. Every Task 3 decision must
# already be frozen before this runs.
!python task3/evaluate_sketch.py --config task3/configs/base.yaml --set {D3}

## 8. Task 4 - Open-Set Recognition

About 6 GPU-hours, the longest task. PROSER initializes from the selected Vanilla checkpoint, so that ordering is mandatory.

In [ ]:
D4 = (f"output.checkpoint_dir={DRIVE}/checkpoints/task4 "
      f"output.results_dir={DRIVE}/results/task4 "
      f"output.cache_dir={DRIVE}/results/task4/cache")

!python task4/train.py --config task4/configs/vanilla.yaml --set {D4}
!python task4/train.py --config task4/configs/gcsc.yaml    --set {D4}
!python task4/train.py --config task4/configs/proser.yaml  --set {D4}

In [ ]:
# Cache logits and features once, so all four scores see identical data.
for m in ["vanilla", "gcsc", "proser"]:
    !python task4/extract_outputs.py --config task4/configs/{m}.yaml --set {D4}

!python task4/evaluate_osr.py --config task4/configs/vanilla.yaml --set {D4}

## 9. Task 1 - Inductive Biases

About 2 GPU-hours. No training dependency, so it can run at any point.

AdaIN weights (~110 MB) download on first use.

In [ ]:
T1 = f"output.results_dir={DRIVE}/results/task1 output.figures_dir={DRIVE}/figures/task1"

!python task1/data/make_subset.py --config task1/configs/base.yaml
!python task1/data/make_cue_conflicts.py --config task1/configs/interventions.yaml
!python task1/scripts/run_task1.py --config task1/configs/interventions.yaml --set {T1}

In [ ]:
# Check the cue-conflict acceptance counts BEFORE trusting shape bias:
# a high shape-bias value resting on few decisions is weak evidence.
import json
print(json.dumps(json.load(open('task1/data/generated/cue_conflict/summary.json')), indent=2))

## 10. Sync results back

Checkpoints stay on Drive. Only the small machine-readable results belong in git.

In [ ]:
# Each task's results belong under <task>/results/, which is where its
# config writes them. Copying {DRIVE}/results/* into the repo root instead
# lands them at <task>/ and splits every task's output across two places.
for t in ["task1", "task2", "task3", "task4"]:
    !mkdir -p /content/pa1/{t}/results /content/pa1/figures/{t}
    !cp -r {DRIVE}/results/{t}/* /content/pa1/{t}/results/ 2>/dev/null
    !cp -r {DRIVE}/figures/{t}/* /content/pa1/figures/{t}/ 2>/dev/null

# Figures the evaluation scripts do not emit, built from the copied CSVs.
!cd /content/pa1 && python scripts/make_figures.py

!cd /content/pa1 && git config user.name "Talha" && \n  git config user.email "tehbenbat5@gmail.com"
!cd /content/pa1 && git add task1/results task2/results task3/results \n  task4/results figures && \n  git commit -m "Add experiment results" && git push

---

## Notes

**Disconnects.** Every training script saves its best checkpoint whenever validation improves, so a disconnect costs at most the epochs since the last improvement.

**Compute discipline.** Smoke tests may be small. Final runs must follow the assignment protocol - do not quietly reduce epochs, data, or the controlled-study grid. If a real constraint forces a change, record it in `progress.md` *before* changing anything, and report it as a limitation.

**Information boundaries.** Enforced in code, but worth knowing: Task 2 never selects on target labels, Task 3 never loads Sketch outside `evaluate_sketch.py`, and Task 4 calibrates thresholds on CIFAR-10 validation only.